<a href="https://colab.research.google.com/github/tawheedul-islam/iot-botnet-brb/blob/main/IoT_BRB_Part7a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 7a: দ্বিতীয় dataset (IoT-23) — আগে শুধু দেখে নেওয়া
**লক্ষ্য:** paper শক্তিশালী করতে N-BaIoT ছাড়া আরেকটা dataset-এ একই পরীক্ষা করা যায় কি না যাচাই। এই notebook **কোনো model চালায় না**। শুধু IoT-23-এর ২০টা capture থেকে প্রতিটার প্রথম ~৬০ MB নামায়, পড়ে, আর দেখায়:
- কয়টা সারি, কতগুলো Benign/Malicious
- কোন ধরনের আক্রমণ (detailed label)
- কোন feature-এ কত শতাংশ মান আসলে ফাঁকা (Zeek-এ `-`), মানে স্বাভাবিকভাবেই missing

**Dataset:** Stratosphere Lab, IoT-23 (CC-BY)। Parmisano, Garcia, Erquiaga, "A labeled dataset with malicious and benign IoT network traffic"।

⏱ প্রায় ৫-১০ মিনিট (ইন্টারনেট গতির উপর)। Runtime → Run all। শেষে **File → Download → .ipynb** করে আমাকে পাঠাও।

In [12]:
import os, io, time, requests
import numpy as np, pandas as pd
BASE = "https://mcfp.felk.cvut.cz/publicDatasets/IoT-23-Dataset/IndividualScenarios/"
LIMIT_MB = 60          # প্রতিটা capture-এর প্রথম এত MB নেওয়া হবে
CAPTURES = {           # capture -> (ধরন, নোট)
 "CTU-IoT-Malware-Capture-34-1": "Mirai",
 "CTU-IoT-Malware-Capture-7-1":  "Mirai",
 "CTU-IoT-Malware-Capture-44-1": "Mirai",
 "CTU-IoT-Malware-Capture-20-1": "Torii",
 "CTU-IoT-Malware-Capture-21-1": "Torii",
 "CTU-IoT-Malware-Capture-8-1":  "Hakai",
 "CTU-IoT-Malware-Capture-3-1":  "Muhstik",
 "CTU-IoT-Malware-Capture-1-1":  "HideAndSeek",
 "CTU-IoT-Malware-Capture-42-1": "Trojan",
 "CTU-IoT-Malware-Capture-9-1":  "Hajime",
 "CTU-IoT-Malware-Capture-36-1": "Okiru",
 "CTU-IoT-Malware-Capture-33-1": "Kenjiro",
 "CTU-IoT-Malware-Capture-60-1": "Gafgyt",
 "CTU-Honeypot-Capture-4-1": "benign (Philips HUE)",
 "CTU-Honeypot-Capture-5-1": "benign (Amazon Echo)",
 "CTU-Honeypot-Capture-7-1": "benign (Somfy doorlock)",
}
os.makedirs("iot23", exist_ok=True)

In [13]:
def download_head(cap, limit_mb=LIMIT_MB):
    url = f"{BASE}{cap}/bro/conn.log.labeled"
    out = f"iot23/{cap}.conn.log.labeled"
    if os.path.exists(out) and os.path.getsize(out) > 0:
        return out, os.path.getsize(out)
    limit = limit_mb * 1024 * 1024
    r = requests.get(url, stream=True, headers={"Range": f"bytes=0-{limit-1}"}, timeout=60)
    r.raise_for_status()
    n = 0
    with open(out, "wb") as f:
        for chunk in r.iter_content(1 << 20):
            f.write(chunk); n += len(chunk)
            if n >= limit: break
    r.close()
    return out, n

t0 = time.time(); got = {}
for cap in CAPTURES:
    try:
        p, n = download_head(cap); got[cap] = p
        print(f"{cap}: {n/1e6:.1f} MB  ({time.time()-t0:.0f}s)")
    except Exception as e:
        print(f"{cap}: ব্যর্থ -> {type(e).__name__}: {str(e)[:120]}")

CTU-IoT-Malware-Capture-34-1: 2.9 MB  (0s)
CTU-IoT-Malware-Capture-7-1: 62.9 MB  (0s)
CTU-IoT-Malware-Capture-44-1: 0.0 MB  (0s)
CTU-IoT-Malware-Capture-20-1: 0.4 MB  (0s)
CTU-IoT-Malware-Capture-21-1: 0.4 MB  (0s)
CTU-IoT-Malware-Capture-8-1: 1.4 MB  (0s)
CTU-IoT-Malware-Capture-3-1: 24.4 MB  (0s)
CTU-IoT-Malware-Capture-1-1: 62.9 MB  (0s)
CTU-IoT-Malware-Capture-42-1: 0.6 MB  (0s)
CTU-IoT-Malware-Capture-9-1: 62.9 MB  (0s)
CTU-IoT-Malware-Capture-36-1: 62.9 MB  (0s)
CTU-IoT-Malware-Capture-33-1: 62.9 MB  (0s)
CTU-IoT-Malware-Capture-60-1: 62.9 MB  (0s)
CTU-Honeypot-Capture-4-1: 0.1 MB  (0s)
CTU-Honeypot-Capture-5-1: 0.2 MB  (0s)
CTU-Honeypot-Capture-7-1: ব্যর্থ -> HTTPError: 404 Client Error: Not Found for url: https://mcfp.felk.cvut.cz/publicDatasets/IoT-23-Dataset/IndividualScenarios/CTU-Hon


In [14]:
import re
def parse_conn(path):
    # Zeek conn.log.labeled পড়ে DataFrame বানায়; অসম্পূর্ণ লাইন বাদ যায়
    names, ncol, rows = None, 0, []
    with open(path, "r", errors="replace") as f:
        for ln in f:
            ln = ln.rstrip("\n")
            if ln.startswith("#fields"):
                hdr = ln.split("\t")[1:]; ncol = len(hdr)
                # শেষ কলামটি বাদ দিয়ে বাকি অংশ নিচ্ছি
                base_cols = hdr[:-1]
                # 'tunnel_parents' এবং 'label' ইতিমধ্যেই base_cols-এ থাকলে কেবল 'detailed_label' যোগ করব,
                # অন্যথায় ৩টি কলামই ম্যানুয়ালি যোগ করব
                if "tunnel_parents" in base_cols and "label" in base_cols:
                    names = base_cols + ["detailed_label"]
                else:
                    # যদি কোনো কারণে আগেরগুলোতে না থাকে, তাহলে নিশ্চিত করার জন্য ৩টিই যোগ করছি
                    names = [c for c in base_cols if c not in ["tunnel_parents", "label"]] + ["tunnel_parents", "label", "detailed_label"]
            elif ln and not ln.startswith("#"):
                parts = ln.split("\t")
                if len(parts) != ncol: continue
                tail = re.split(r"\s+", parts[-1].strip(), maxsplit=2)
                if len(tail) != 3: continue
                rows.append(parts[:-1] + tail)
    assert names, "#fields লাইন পাওয়া যায়নি"
    # ডাটা রো-র কলাম সংখ্যার সাথে হেডার নামের কলাম সংখ্যা সামঞ্জস্যপূর্ণ কি না তা নিশ্চিত করছি
    final_rows = [r for r in rows if len(r) == len(names)]
    return pd.DataFrame(final_rows, columns=names)

NUM = ["duration", "orig_bytes", "resp_bytes", "orig_pkts", "orig_ip_bytes", "resp_pkts", "resp_ip_bytes"]
summary = []
frames = {}
for cap, p in got.items():
    d = parse_conn(p)
    for c in NUM: d[c] = pd.to_numeric(d[c].replace("-", np.nan), errors="coerce")
    frames[cap] = d
    d[["proto", "service", "conn_state"] + NUM + ["label", "detailed_label"]].to_csv(f"iot23/{cap}.csv", index=False)
    vc = d["label"].value_counts()
    summary.append(dict(capture=cap, kind=CAPTURES[cap], rows=len(d),
                        benign=int(vc.get("Benign", 0)), malicious=int(vc.get("Malicious", 0))))
S = pd.DataFrame(summary); print(S.to_string(index=False))

                     capture                 kind   rows  benign  malicious
CTU-IoT-Malware-Capture-34-1                Mirai      0       0          0
 CTU-IoT-Malware-Capture-7-1                Mirai 455777     601     455176
CTU-IoT-Malware-Capture-44-1                Mirai    237     211         26
CTU-IoT-Malware-Capture-20-1                Torii   3209    3193         16
CTU-IoT-Malware-Capture-21-1                Torii   3286    3272         14
 CTU-IoT-Malware-Capture-8-1                Hakai  10403    2181       8222
 CTU-IoT-Malware-Capture-3-1              Muhstik 156103    4536     151567
 CTU-IoT-Malware-Capture-1-1          HideAndSeek 426890  194631     232259
CTU-IoT-Malware-Capture-42-1               Trojan   4426    4420          6
 CTU-IoT-Malware-Capture-9-1               Hajime 403971    2258     401713
CTU-IoT-Malware-Capture-36-1                Okiru 480165     104     480061
CTU-IoT-Malware-Capture-33-1              Kenjiro 435447   11065     424382
CTU-IoT-Malw

In [15]:
print("===== প্রতিটা capture: top detailed label =====")
for cap, d in frames.items():
    print(cap, CAPTURES[cap], "->", d["detailed_label"].value_counts().head(4).to_dict())

print("\n===== কত শতাংশ মান ফাঁকা (NaN), সব capture মিলিয়ে =====")
allf = pd.concat(frames.values(), ignore_index=True)
print((allf[NUM].isna().mean() * 100).round(1).to_string())

print("\n===== Benign বনাম Malicious আলাদা করে ফাঁকা % =====")
print(allf.groupby("label")[NUM].apply(lambda g: g.isna().mean() * 100).round(1).to_string())

print("\n===== feature-এর পরিসর (Benign বনাম Malicious, মধ্যমা) =====")
print(allf.groupby("label")[NUM].median().round(3).to_string())

===== প্রতিটা capture: top detailed label =====
CTU-IoT-Malware-Capture-34-1 Mirai -> {}
CTU-IoT-Malware-Capture-7-1 Mirai -> {'Okiru': 454659, '-': 601, 'C&C-HeartBeat': 517}
CTU-IoT-Malware-Capture-44-1 Mirai -> {'-': 211, 'C&C': 14, 'C&C-FileDownload': 11, 'DDoS': 1}
CTU-IoT-Malware-Capture-20-1 Torii -> {'-': 3193, 'C&C-Torii': 16}
CTU-IoT-Malware-Capture-21-1 Torii -> {'-': 3272, 'C&C-Torii': 14}
CTU-IoT-Malware-Capture-8-1 Hakai -> {'C&C': 8222, '-': 2181}
CTU-IoT-Malware-Capture-3-1 Muhstik -> {'PartOfAHorizontalPortScan': 145597, 'Attack': 5962, '-': 4536, 'C&C': 8}
CTU-IoT-Malware-Capture-1-1 HideAndSeek -> {'PartOfAHorizontalPortScan': 232253, '-': 194631, 'C&C': 6}
CTU-IoT-Malware-Capture-42-1 Trojan -> {'-': 4420, 'FileDownload': 3, 'C&C-FileDownload': 3}
CTU-IoT-Malware-Capture-9-1 Hajime -> {'PartOfAHorizontalPortScan': 401713, '-': 2258}
CTU-IoT-Malware-Capture-36-1 Okiru -> {'Okiru': 479507, 'C&C-HeartBeat': 551, '-': 104, 'Okiru-Attack': 3}
CTU-IoT-Malware-Capture-33-1

---
## এরপর
**File → Download → .ipynb** করে attach করো। আমি এটা দেখে ঠিক করব:
1. কোন capture দিয়ে train, কোন capture-এ test (নতুন ম্যালওয়্যার ধরন)
2. কোন ৬টা feature ব্যবহার হবে
3. IoT-23 আদৌ উপযুক্ত কি না (যেমন benign নমুনা যথেষ্ট কি না, device আর label মিশে গেছে কি না)